# PYNQ-Z2 螺丝识别与分拣

本 Notebook 使用共享 C 核心与 PYNQ 平台适配层。先运行识别与坐标检查，再显式启用动作。末尾关闭会等待 AXI/DMA 排空。

In [ ]:
from pathlib import Path
import sys, json, time

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/python/screwsortmint/app.py').is_file()), None)
if root is None:
    raise RuntimeError('请从解压后的项目目录或其子目录打开 Notebook')
sys.path.insert(0, str(root/'src/python'))
from screwsortmint import ScrewSortMint
print(root)


## 编译与来源核验

此单元只核验文件与编译库，不加载 Overlay。

In [ ]:
import subprocess
subprocess.run([sys.executable, str(root/'build/verify.py')], check=True)
subprocess.run([sys.executable, str(root/'build/build.py'), 'linux'], check=True)
print('Shared MC18 application ready')


## 启动识别

不启用机械臂。每个硬件会话创建新的对象，关闭之前不要重新运行本单元。

In [ ]:
app = ScrewSortMint()
try:
    app.prepare().initialize().start()
except BaseException:
    app.close()
    raise
time.sleep(1)
print(app.status())
print(app.results())
print(app.messages())


In [ ]:
print(app.results())
print(app.metrics())
print(app.messages())


## 显式启用动作

先核对平台坐标、拾取高度和四料框实际位置。螺母仅识别。开启后板上摇杆也能发起动作。

In [ ]:
# 核对完成后，移除下一行的注释并执行。
# app.enable_motion()


## 选择一种操作

一次执行一种操作。自动分拣最多四种螺丝，动作期间暂停识别，后续抓取要求新帧和移除确认。

In [ ]:
# app.select_direction('right')
# app.pick()

# app.auto_pick('M3_1')

# app.sort_all()

print(app.status())
print(app.messages())


## 停止、保存现场与关闭

停止优先于队列中的动作；关闭成功后才会释放连续内存。

In [ ]:
app.stop()
print(app.status())
print(app.messages())


In [ ]:
from datetime import datetime, timezone
out = root/'board/output'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
out.mkdir(parents=True, exist_ok=True)
record = dict(status=app.status(), results=app.results(), metrics=app.metrics(), messages=app.messages())
(out/'notebook_observation.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
print(out)


In [ ]:
try:
    app.stop()
finally:
    app.close()
print('会话已关闭；下次启动创建新的 ScrewSortMint 对象。')
